# LTIC week 1 on Pl@ntNet-300K (Kaggle)

Plan v9.1 week 1: split files and V-parts, taxonomy, chunked features, LA probe, finding C-F,
the week-1 gate diagnostics, and conformal sets. Optional: baselines 1-9. Full guide: `KAGGLE_WEEK1.md`.

Right-hand panel before running:
1. **Accelerator:** `GPU T4 x2`
2. **Internet:** On (code, encoder weights, GBIF)
3. **Input:** the dataset **plantnet 300k images** (`noahbadoa/plantnet-300k-images`)

Then **Save Version -> Save & Run All (Commit)**. It keeps running after you close the browser.

In [ ]:
# ---- Settings: the only cell you need to edit ----
GITHUB_USER = "dgpro06"
BRANCH      = "Dgpro"

# Session 1 (the gate, ~1.5-3 h): ENCODERS = "clip_b32", RUN_GATE = True, BASELINES = ""
# Session 2 (overnight): ENCODERS = "clip_b16,bioclip2,dinov2_b14" or BASELINE_ENCODERS = "clip_b32",
#   BASELINES = the list below, PREV = the folder of session 1's Output (attach it as an input first)
ENCODERS          = "clip_b32"     # comma-separated: clip_b32,clip_b16,bioclip2,dinov2_b14
RUN_GATE          = True           # LA probe + C-F + gate diagnostics + conformal on the first encoder
BASELINES         = ""             # e.g. "ce,balanced_softmax,crt,tau_norm,ldam_drw,reslt,ltic2,ltic3,ltic3@gamma=0,flat_genus"
BASELINE_ENCODERS = ""             # default: ENCODERS
TRIALS            = 20             # per method; cut to 10 for ALL methods if one run takes > 2 min
PREV              = ""             # e.g. "/kaggle/input/ltic-week-1" (an earlier version's Output) to resume

## Step 1: check the GPU and the inputs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /kaggle/input

## Step 2: get the code from GitHub

In [ ]:
import os
CODE = "/kaggle/working/LTIC-Herb"
if os.path.isdir(CODE):
    !git -C {CODE} pull
else:
    !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/LTIC-Herb.git {CODE}
!git -C {CODE} log --oneline -1

## Step 3: run the week-1 pipeline (`kaggle/week1_kaggle.sh`)

In [ ]:
os.environ.update(ENCODERS=ENCODERS, RUN_GATE="1" if RUN_GATE else "0", BASELINES=BASELINES,
                  BASELINE_ENCODERS=BASELINE_ENCODERS or ENCODERS, TRIALS=str(TRIALS), PREV=PREV)
!bash {CODE}/kaggle/week1_kaggle.sh
if get_ipython().user_ns.get("_exit_code", 0) != 0:
    raise RuntimeError("The run stopped with an error - scroll up to the first error message.")

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob("/kaggle/working/results/week1_gate_*.json")):
    g = json.load(open(path))
    print(path)
    for k, v in g["checks"].items():
        print(f"  {k:28s} {v['value']:.4f}  {'PASS' if v['pass'] else 'FAIL'}")
    print("  verdict:", g["verdict"])
print(json.load(open("/kaggle/working/splits/plantnet_vparts.json"))["decision_8_0"])
!ls -R /kaggle/working/results | head -50